# pyMVGC Tutorial

This implementation of MVGC, from Barnett and Seth (2014), works by moving between autocovariance sequence and VAR parameters using the Yule-Walker equations as follows:
 1. The model order (maximum lag) of the VAR is estimated through learning VAR parameters using increasing orders and minimising an information criterion score (AIC or BIC).
 2. A VAR of the selected order is estimated from time-series data using OLS.
 3. The VAR parameters and residual covariance matrix are used to calculate an autocovariance sequence representing the VAR, via the reverse Yule-Walker equations.
 4. The full model's residual covariance matrix is estimated from this autocovariance sequence using the Yule-Walker equations. 
 5. The reduced model's residual covariance matrix is estimated from the same autocovariance sequence with the columns and rows related to the potential causal variable(s) (X) removed.
 6. The potential caused (Y) variable(s) are selected from the residual covariance matrices. The Granger causality statistic is the logged ratio of the linear determinants of these selected matrices. This value gives us the Granger causality from X to Y, conditioned on all other variables in the VAR.
---
For the following example, we will work with some data on inflation, unemployment and interest rates in the US:

In [8]:
import pyMVGC as gc
import numpy as np
import statsmodels.api as sm


#load data
mdata = sm.datasets.macrodata.load_pandas().data

#select desired columns
cols = ['infl', 'unemp', 'realint']

idx = {c: i for i, c in enumerate(cols)}
raw = mdata[cols].values
data = (np.diff(raw, axis = 0))

#normalise
data /= data.std(axis = 0, keepdims=True)


## Model order selection

`pyMVGC` provides the following function to select the optimal VAR in the range of `1` to `maxLags`:

In [17]:
aics = gc.estimate_model_order(data=data,
                               maxLag = 10, 
                               criterion = "AIC",
                               verbose = True)


ENTER estimate_model_order id=1781875035.8979254
AIC at lag 1 = -641.5556544126996
AIC at lag 2 = -668.7204346318608
AIC at lag 3 = -657.5402674356843
AIC at lag 4 = -665.280078327037
AIC at lag 5 = -661.359255735935
AIC at lag 6 = -638.1390724361501
AIC at lag 7 = -618.178271995806
AIC at lag 8 = -605.9031677938581
AIC at lag 9 = -572.363950007001
AIC at lag 10 = -521.2344292355975
Minimal AIC at lag 2


We can see that the optimal VAR order for our data based on AIC is 2.

---

## Granger causality estimation


Once we have estimated the model order, we can calculate multivariate Granger causality using a single funciton. 

`ts_to_pMVGC` outputs a matrix containnig the Granger causality from variables in columns to variables in rows, conditioned on all other variables:


In [10]:
gc.ts_to_pMVGC(data = data, p = 2)

array([[       nan, 0.0241427 , 0.02044804],
       [0.02341437,        nan, 0.0145652 ],
       [0.01708623, 0.01350842,        nan]])

Alternatively, we can calculate the Granger causality from variables at `to_idx` to `from_idx`. Note that these can contain multiple indexes, passed in a list. For example, to calculate the joint Granger causality from inflation and unemployment on interest rates:

In [11]:
gc.ts_to_MVGC(data = data,
              p = 2, 
              to_idx = idx["realint"],
              from_idx = [idx["infl"], idx["unemp"]])

0.03517445693329957

---
The modules performing the steps outlined in the introductiona are available, allowing the user more flexibilty:

In [12]:
 # Learn VAR from time series data.
Phi, resCov = gc.ts_to_var(data = data, p = 2)

# Learn autocovariance sequence from VAR.
autocov = gc.var_to_autocov(Phi = Phi, resCovMat = resCov) 

At this stage, Granger causality can be calculated manually via `autocov_to_MVGC` or `autocov_to_pMVGC`:

In [13]:
# Get multivariate Granger causality from from autocovarance sequence.
GC = gc.autocov_to_MVGC(autocov, 
                        to_idx = idx["realint"],
                        from_idx = [idx["infl"], idx["unemp"]]) 

# Get pairwise Granger causality from autocovariance sequence.
pGC = gc.autocov_to_pMVGC(autocov) 

Alternatively, to calculate Granger caulity manually, the VAR parameters for the full and reduced regression can be obtained from `autocov_to_var`:


In [14]:
Phi, autocov = gc.autocov_to_var(autocov = autocov)